# Understanding NDCG (Normalized Discounted Cumulative Gain)

This tutorial walks through how NDCG is computed from scratch, then verifies the result using PyTorch Ignite's `Ndcg` metric.

NDCG is a ranking metric commonly used in information retrieval and recommender systems. Unlike metrics that only check if the right item was retrieved, NDCG rewards models that rank more relevant items **higher** in the list.

By the end of this notebook you will:
- Understand what ground truth and predictions look like for a ranking problem
- Compute DCG and IDCG step by step by hand
- Calculate NDCG manually
- Verify every number matches the Ignite `Ndcg` implementation

## Setup

In [1]:
# Install dependencies if needed
# !pip install pytorch-ignite torch
import torch
import math

## 1. The Problem Setup

Imagine a search engine returning 5 documents for a query. Each document has a **relevance score** (ground truth) assigned by a human — higher means more relevant:

| Document | Relevance (ground truth) |
|----------|-------------------------|
| Doc A    | 3 (highly relevant)     |
| Doc B    | 2 (relevant)            |
| Doc C    | 3 (highly relevant)     |
| Doc D    | 0 (not relevant)        |
| Doc E    | 1 (slightly relevant)   |

The model predicts a **score** for each document. The model then ranks documents by these scores (highest score = rank 1):

In [2]:
# Ground truth relevance scores (one query, 5 documents)
# Shape: (1, 5) — batch of 1 query
y_true = torch.tensor([[3.0, 2.0, 3.0, 0.0, 1.0]])

# Model prediction scores for each document
# Higher score = model thinks this doc is more relevant
y_pred = torch.tensor([[0.1, 0.4, 0.35, 0.8, 0.1]])

print("Ground truth relevance:", y_true)
print("Model prediction scores:", y_pred)

Ground truth relevance: tensor([[3., 2., 3., 0., 1.]])
Model prediction scores: tensor([[0.1000, 0.4000, 0.3500, 0.8000, 0.1000]])


## 2. Step 1 — Rank the Documents by Model Score

The model ranks documents by sorting its predicted scores in descending order. The document with the highest predicted score gets rank 1.

In [3]:
# Sort document indices by predicted score (descending)
ranked_indices = torch.argsort(y_pred, descending=True)
print("Ranked document indices (by model):", ranked_indices)

# Reorder ground truth relevance scores according to model ranking
ranked_relevance = y_true[0][ranked_indices[0]]
print("Relevance scores in model's ranked order:", ranked_relevance)
print()
print("So the model ranked:")
doc_names = ['Doc A', 'Doc B', 'Doc C', 'Doc D', 'Doc E']
for rank, idx in enumerate(ranked_indices[0]):
    print(f"  Rank {rank+1}: {doc_names[idx]} (relevance={y_true[0][idx].item():.0f}, pred score={y_pred[0][idx].item():.2f})")

Ranked document indices (by model): tensor([[3, 1, 2, 0, 4]])
Relevance scores in model's ranked order: tensor([0., 2., 3., 3., 1.])

So the model ranked:
  Rank 1: Doc D (relevance=0, pred score=0.80)
  Rank 2: Doc B (relevance=2, pred score=0.40)
  Rank 3: Doc C (relevance=3, pred score=0.35)
  Rank 4: Doc A (relevance=3, pred score=0.10)
  Rank 5: Doc E (relevance=1, pred score=0.10)


## 3. Step 2 — Compute DCG (Discounted Cumulative Gain)

DCG measures the quality of the ranking. It rewards relevant documents but **discounts** them based on their position — finding a relevant document at rank 1 is worth more than finding it at rank 5.

The formula is:

$$DCG@K = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i + 1)}$$

Where:
- $rel_i$ is the relevance of the document at rank $i$
- The numerator $2^{rel_i} - 1$ is the **gain** (higher relevance = exponentially higher gain)
- The denominator $\log_2(i+1)$ is the **discount** (lower rank position = smaller discount)

In [4]:
K = 5  # We evaluate the top 5 results

dcg = 0.0
print(f"Computing DCG@{K} step by step:\n")
print(f"{'Rank':<6} {'Doc':<8} {'Relevance':<12} {'Gain (2^rel-1)':<18} {'Discount log2(i+1)':<22} {'Contribution'}")
print("-" * 80)

for i, idx in enumerate(ranked_indices[0][:K]):
    rank = i + 1
    rel = y_true[0][idx].item()
    gain = (2 ** rel) - 1
    discount = math.log2(rank + 1)
    contribution = gain / discount
    dcg += contribution
    print(f"{rank:<6} {doc_names[idx]:<8} {rel:<12.0f} {gain:<18.4f} {discount:<22.4f} {contribution:.4f}")

print("-" * 80)
print(f"DCG@{K} = {dcg:.4f}")

Computing DCG@5 step by step:

Rank   Doc      Relevance    Gain (2^rel-1)     Discount log2(i+1)     Contribution
--------------------------------------------------------------------------------
1      Doc D    0            0.0000             1.0000                 0.0000
2      Doc B    2            3.0000             1.5850                 1.8928
3      Doc C    3            7.0000             2.0000                 3.5000
4      Doc A    3            7.0000             2.3219                 3.0147
5      Doc E    1            1.0000             2.5850                 0.3869
--------------------------------------------------------------------------------
DCG@5 = 8.7944


## 4. Step 3 — Compute IDCG (Ideal DCG)

IDCG is the DCG of the **perfect ranking** — what score would we get if the model ranked documents in the exact order of their true relevance?

We compute this by sorting the ground truth relevance scores in descending order and computing DCG on that ideal ordering.

In [5]:
# The ideal ranking: sort ground truth relevance descending
ideal_relevance, _ = torch.sort(y_true[0], descending=True)
print("Ideal relevance order:", ideal_relevance)

idcg = 0.0
print(f"\nComputing IDCG@{K} step by step:\n")
print(f"{'Rank':<6} {'Relevance':<12} {'Gain (2^rel-1)':<18} {'Discount log2(i+1)':<22} {'Contribution'}")
print("-" * 72)

for i in range(K):
    rank = i + 1
    rel = ideal_relevance[i].item()
    gain = (2 ** rel) - 1
    discount = math.log2(rank + 1)
    contribution = gain / discount
    idcg += contribution
    print(f"{rank:<6} {rel:<12.0f} {gain:<18.4f} {discount:<22.4f} {contribution:.4f}")

print("-" * 72)
print(f"IDCG@{K} = {idcg:.4f}")

Ideal relevance order: tensor([3., 3., 2., 1., 0.])

Computing IDCG@5 step by step:

Rank   Relevance    Gain (2^rel-1)     Discount log2(i+1)     Contribution
------------------------------------------------------------------------
1      3            7.0000             1.0000                 7.0000
2      3            7.0000             1.5850                 4.4165
3      2            3.0000             2.0000                 1.5000
4      1            1.0000             2.3219                 0.4307
5      0            0.0000             2.5850                 0.0000
------------------------------------------------------------------------
IDCG@5 = 13.3472


## 5. Step 4 — Compute NDCG

NDCG normalizes DCG by IDCG, giving a score between 0 and 1:

$$NDCG@K = \frac{DCG@K}{IDCG@K}$$

A score of 1.0 means the model ranked everything perfectly. A score close to 0 means the ranking was very poor.

In [6]:
ndcg_manual = dcg / idcg

print(f"DCG@{K}  = {dcg:.4f}")
print(f"IDCG@{K} = {idcg:.4f}")
print(f"NDCG@{K} = DCG / IDCG = {dcg:.4f} / {idcg:.4f} = {ndcg_manual:.4f}")
print(f"\nThe model achieved {ndcg_manual*100:.1f}% of the ideal ranking quality.")

DCG@5  = 8.7944
IDCG@5 = 13.3472
NDCG@5 = DCG / IDCG = 8.7944 / 13.3472 = 0.6589

The model achieved 65.9% of the ideal ranking quality.


## 6. Verify with PyTorch Ignite

Now let's confirm our manual calculation matches the Ignite `Ndcg` metric exactly.

In [7]:
from ignite.metrics.rec_sys.ndcg import NDCG

# Initialize the Ndcg metric with k=5
ndcg_metric = NDCG(output_transform=lambda x: x, top_k=[K])

# Reset and update with our data
ndcg_metric.reset()
ndcg_metric.update((y_pred, y_true))

# Compute the result
ignite_result = ndcg_metric.compute()

print(f"Manual NDCG@{K}:  {ndcg_manual:.4f}")
print(f"Ignite NDCG@{K}:  {ignite_result[0]:.4f}")
print()

# Verify they match
assert abs(ndcg_manual - ignite_result[0]) < 1e-4, "Mismatch!"
print("✓ Manual calculation matches Ignite implementation perfectly!")

Manual NDCG@5:  0.6589
Ignite NDCG@5:  0.6589

✓ Manual calculation matches Ignite implementation perfectly!


## 7. Understanding the Score

Let's build some intuition by looking at two extreme cases.

In [8]:
# Case 1: Perfect ranking (model scores match relevance exactly)
y_pred_perfect = torch.tensor([[0.9, 0.6, 0.8, 0.1, 0.3]])

ndcg_metric.reset()
ndcg_metric.update((y_pred_perfect, y_true))
perfect_score = ndcg_metric.compute()
print(f"Perfect ranking NDCG@{K}: {perfect_score[0]:.4f} (should be 1.0)")

# Case 2: Worst ranking (model ranks least relevant items highest)
y_pred_worst = torch.tensor([[0.1, 0.3, 0.2, 0.9, 0.6]])

ndcg_metric.reset()
ndcg_metric.update((y_pred_worst, y_true))
worst_score = ndcg_metric.compute()
print(f"Worst ranking  NDCG@{K}: {worst_score[0]:.4f} (should be close to 0)")

print(f"\nOur model's ranking NDCG@{K}: {ndcg_manual:.4f} (somewhere in between)")

Perfect ranking NDCG@5: 1.0000 (should be 1.0)
Worst ranking  NDCG@5: 0.5884 (should be close to 0)

Our model's ranking NDCG@5: 0.6589 (somewhere in between)
